# MNDWI site thresholds
For each site across the full date range:
* Load in low cloud images (< 1% SCL ignore)
* Calculate the mean MNDWI within the ROI
* Record the low 30% threshold

In [ ]:
import sys
import pathlib
import geopandas
import numpy

import pandas

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
    
import utils
import sentinel2

%load_ext autoreload
%autoreload 2

In [2]:
data_path = utils.get_data_path()
site_polygons_path = data_path / "site_polygons"
output_file = data_path / "mndwi_low_tide_thresholds.csv"

## Sites, years and cloud settings

In [3]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
sites = all_training_sites

years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

max_cloud_cover = 1

In [ ]:
_, satellite_path = utils.get_prediction_path(
    sample_method="sampling_2", method_2_threshold=0.5,
    model_max_cloud_cover=max_cloud_cover,
    predict_max_cloud_cover=max_cloud_cover,
    satellite_only=True) 

# Save satellite images

In [ ]:
# Path to downlaoded satellites
print(satellite_path)

for site in sites:
    for year in years:
        print(f"{site}: year {year}")
    
        satellite_file = satellite_path / f"{site}_{year}_sentinel-2.nc"
        site_polygon_file = utils.get_site_polygon_path(site)

        if (satellite_path / f"{site}_{years[-1]}_sentinel-2.nc").exists():
            print(f"\tAlready run for {site} as {site}_{years[-1]}_sentinel-2.nc exists. "
                  "Delete if you want redownloaded.")
            break
    
        if not satellite_file.exists():
            print(f"\tSave satellite image of the site {site} with < {max_cloud_cover}% cloud in the year {year}")
            site_polygon = geopandas.read_file(site_polygon_file)
            satellite_data = sentinel2.get_cloud_filtered_images_in_year(
                geometry=site_polygon,
                site_name=site, # todo pass this and use to lookup the site_polygon internally
                max_cloud_cover=max_cloud_cover,
                year=year,
                apply_anomaly_filter=False,
            )
            if len(satellite_data['time']) == 0:
                print("Warning: No satellite data without cloud. Ignore")
            else:
                print("\t\tSave")
                utils.write_netcdf_conventions_in_place(satellite_data)
                satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                utils.write_netcdf_conventions_in_place(satellite_data)
                utils.save_netcdf(satellite_data, satellite_file)
            utils.rgb_from_satellite(satellite_data_path=satellite_file)

C:\Local\repos\seagrass-detection\data\predictions\cloud_1\satellite_images
Ihutai: year 2016
	Already run for Ihutai as Ihutai_2026_sentinel-2.nc exists. Delete if you want redownloaded.
CatlinsLake: year 2016
	Already run for CatlinsLake as CatlinsLake_2026_sentinel-2.nc exists. Delete if you want redownloaded.
CatlinsRiverMouth: year 2016
	Already run for CatlinsRiverMouth as CatlinsRiverMouth_2026_sentinel-2.nc exists. Delete if you want redownloaded.
Childrens: year 2016
Childrens: year 2017
Childrens: year 2018
Childrens: year 2019
Childrens: year 2020
	Save satellite image of the site Childrens with < 1% cloud in the year 2020
	Getting images
	Low cloud tiles: 31 from the total tiles of 143. Cloud percentages [100.  100.  100.  100.    0.    8.8   0.1  99.5  22.7 100.  100.  100.
 100.   66.1 100.    0.  100.  100.    0.  100.    0.   30.    0.5 100.
   0.    1.7 100.    2.4   3.1 100.  100.    0.    0.  100.  100.    3.2
 100.  100.   26.6  10.5   4.8  18.7   0.    0.    0.  10

# Calculate MNDWI, Mean MNDWI lower 20% threshold

In [ ]:
mndwi_thresholds = {"site": [], "threshold": []}
mean_mndwi_all_sites = []
sites = sites[0:2]

for site in sites[0:2]:
    mean_mndwi_series = []
    for year in years:
        print(f"{site}: year {year}")
    
        satellite_file = satellite_path / f"{site}_{year}_sentinel-2.nc"

        # load in satellite file
        satellite_data = utils.load_satellite(filename=satellite_file, chunks=None)
        # calculate MNDWI layer
        green = satellite_data["B03"].astype(numpy.float32)
        swir1 = satellite_data["B11"].astype(numpy.float32)
        scl = satellite_data["SCL"]

        valid_pixels = (
            scl.notnull()
            & (scl > 0)
            & ~scl.isin(sentinel2.SCL_TO_IGNORE)
            & (green > 0)
            & (swir1 > 0)
        )

        mndwi = ((green - swir1) / (green + swir1)).where(valid_pixels)
        mean_mndwi_series.append(
            mndwi.mean(dim=["x", "y"], skipna=True).compute().to_pandas()
        )
    mean_mndwi_series = pandas.concat(mean_mndwi_series)
    mndwi_thresholds["site"].append(site)  
    mndwi_thresholds["threshold"].append(mean_mndwi_series.quantile(0.20))
    mean_mndwi_all_sites.append(mean_mndwi_series)
mndwi_thresholds = pandas.DataFrame(mndwi_thresholds)
mndwi_thresholds.to_csv(output_file, index=False)

# for plotting the distributions
mean_mndwi_all_sites = pandas.concat(mean_mndwi_all_sites, keys=sites, names=["site", "time"]).rename("mean_mndwi")
mean_mndwi_all_sites.to_csv(satellite_file.with_name("mean_mndwi.csv"))